# Chapter 5 : Reflection
# What is Parallelization ?

Reflection in AI agents is a cognitive design pattern where a model explicitly evaluates its own initial outputs, reasoning, or actions to identify errors and refine them before finalizing a task. It acts as an internal feedback loop, allowing the agent to self-correct rather than simply outputting its first guess.

# Doing API Authentication

In [247]:
from google import genai
from google.genai import types

# Developer TODO: Replace YOUR_API_KEY with your API key.
API_KEY = "YOUR API KEY"

client = genai.Client(
    vertexai=False, api_key=API_KEY
)

In [248]:
import os

os.environ["GEMINI_API_KEY"] = "YOUR API KEY"

In [249]:
chat = client.chats.create(model="gemini-3.6-flash")

In [250]:
from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.tools import google_search
GEMINI_MODEL="gemini-3.6-flash"
import os
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import os



In [251]:
generator = LlmAgent(
    name = "DraftWriter",
    model= GEMINI_MODEL,
    description = "Generates inital draft content on a given subject",
    instruction = "Write a short, informative paragraph about the user's subejct",
    output_key = "draft_key" # The output is saved to this state key
)


In [252]:
# The Second agent critiques the draft from the first agent
reviewer = LlmAgent(
    name = "FactChecker",
    model = GEMINI_MODEL,
    description = "Reviews a given text for factual accuracy and provides a structured critique",
    instruction =  """ You are a meticulous fact-checker.
    1. Read the text provided in the state key 'draft_key
    2. Carefully verify the factual accuracy of all claims
    3. Your Fubak iutput must be a dictionary containing two keys :
    - stattus : A string, eiher Accurate or Inaccurate
    - reasoning : A string prividing a clear explanation for your status, citing specific issues if any are found""",
    output_key = "review_output",
)



In [253]:
# Sequential Agent

review_pipeline = SequentialAgent(
    name = "WriteandReview_pipeline",
    sub_agents = [generator,reviewer]
)

/tmp/ipykernel_4427/3516920085.py:3: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  review_pipeline = SequentialAgent(


In [254]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(review_pipeline)

await runner.session_service.create_session(
    app_name=runner.app_name,
    user_id="user",
    session_id="session"
)


Session(id='session', app_name='InMemoryRunner', user_id='user', state={}, events=[], last_update_time=1790115696.6197422)

In [255]:

from google.genai import types

async for event in runner.run_async(
    user_id="user",
    session_id="session",
    new_message=types.Content(
        role="user",
        parts=[
            types.Part(
                text="WWrite a short, informative paragraph about money laundering."
            )
        ]
    )
):
    if event.is_final_response():
        print(event.content.parts[0].text)

Money laundering is the illegal process of concealing the origins of illicitly obtained funds to make them appear legitimate. This financial crime typically unfolds in three distinct stages: **placement**, where "dirty" money is introduced into the financial system; **layering**, where the funds are moved through complex transactions to obscure their source; and **integration**, where the now "clean" money is reintroduced into the economy for normal use. By enabling organized crime, drug trafficking, and corruption, money laundering poses a severe threat to global financial stability, driving governments and international bodies to enforce strict anti-money laundering (AML) regulations.
```json
{
  "status": "Accurate",
  "reasoning": "The provided text is factually accurate in its definition of money laundering, its description of the three recognized stages (placement, layering, and integration), and its overview of the associated impacts and regulatory responses."
}
```


-- THE END ---